# Phase 3 · Triple Fusion — Scaleup Version (11 Clinical Features)
## PneumoFusionNet | MIMIC-CXR PA | Full Scaleup Dataset (~3,763 PA Chest X-Rays)
### Model Architecture: DenseNet-121 + Bio_ClinicalBERT + 11-Feature Clinical MLP

---

| Item | Value |
|---|---|
| Dataset | `phase3_paired_scaleup_final.csv` (~3,763 rows) |
| Phase 1 ckpt | `Phase_1.1v4_PA_crossval_scaleup/best_model_fold5.pth` |
| Phase 2 ckpt | `Phase_2v2_Scaleup/best_v2_model.pth` |
| Clinical features | **11** (vitals removed — 78.3% missing in non-ICU patients) |
| Baseline to beat | AUC 0.9690 (Phase 3 Scaleup — 17 features) |

---

### 🛡️ Data Leakage Prevention Safeguards
- **Text Leakage**: `IMPRESSION` section excluded; only `FINDINGS` and `HISTORY` used.
- **Imputation Safeguard**: Missing values filled using **training set median only**.
- **Standardisation Safeguard**: `StandardScaler` fitted **exclusively on the training split**.
- **Patient-Level Split**: Stratified 70 / 15 / 15 split.

### ✂️ Why 6 Vital Signs Were Removed
In MIMIC-IV, vital signs (`heart_rate`, `resp_rate`, `spo2`, `systolic_bp`, `diastolic_bp`, `temperature_f`) are recorded via the ICU `chartevents` table only.
Since **78.3% of scaleup patients are non-ICU**, these features had 2,946/3,763 values replaced by constant training medians — introducing **noise, not signal** into the clinical branch.
Removing them and retaining 11 high-coverage lab + demographic features is expected to improve model performance.


---
## Step 1: System Imports & Reproducibility Setup


In [ ]:
# --- CELL 0: Imports & Reproducibility ---
import os, random, warnings, json, copy, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import AutoTokenizer, AutoModel
import torchxrayvision as xrv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, accuracy_score, f1_score
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')


---
## Step 2: Global Configuration & File Paths
**Only change from original scaleup notebook:** `CLINICAL_FEATURES` reduced from 17 to 11 (6 vitals removed), and `SAVE_DIR` updated.


In [ ]:
# --- CELL 1: Configuration & Paths (SCALEUP 11-FEATURE VERSION) ---
# ============================================================
# Dataset : Full MIMIC-CXR PA scaleup (~3,763 images)
# Phase 1 : Phase_1.1v4_PA_crossval_scaleup (fold 5 — best AUC=0.8500)
# Phase 2 : Phase_2v2_Scaleup checkpoint
# Clinical: 11 features (6 noisy vitals removed — 78.3% missing)
# ============================================================

DATASET_DIR  = r'C:\2026\PneumoFusionNet\mimic\main\dataset'
MAIN_DIR     = r'C:\2026\PneumoFusionNet\mimic\main'

# Input data — single merged CSV (text + image + clinical all-in-one)
PAIRED_CSV   = os.path.join(DATASET_DIR, 'phase3_paired_scaleup_final.csv')
BBOX_CSV     = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'lung_bboxes.csv')

# Checkpoints from scaleup runs (IDENTICAL to original scaleup notebook)
P1_CKPT      = os.path.join(MAIN_DIR, 'outputs', 'Phase_1.1v4_PA_crossval_scaleup', 'best_model_fold5.pth')
P2V2_CKPT    = os.path.join(MAIN_DIR, 'outputs', 'Phase_2v2_Scaleup', 'best_v2_model.pth')

# Output — separate folder so original 17-feature results are preserved
SAVE_DIR     = os.path.join(MAIN_DIR, 'outputs', 'Phase_3_triple_fusion_scaleup_11features')
os.makedirs(SAVE_DIR, exist_ok=True)

# Model hyperparams (IDENTICAL to original scaleup notebook)
IMG_SIZE       = 224
MAX_TEXT_LEN   = 256
CLINBERT_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
IMG_FEAT_DIM   = 1024
TXT_FEAT_DIM   = 768
ATTN_DIM       = 512
ATTN_HEADS     = 8
META_HIDDEN    = 128
META_OUT_DIM   = 64

# Training (IDENTICAL to original scaleup notebook)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
BATCH_SIZE  = 16
LR_FUSION   = 2e-4
LR_BERT     = 1e-5
LR_META     = 1e-3
EPOCHS      = 40
PATIENCE    = 8
FOCAL_GAMMA = 2.0
MIXUP_ALPHA = 0.2
CLASSES     = ['NORMAL', 'PNEUMONIA']
MEAN = [0.5020]; STD = [0.2703]

# Image base path (relative paths in CSV are relative to DATASET_DIR)
IMG_BASE = DATASET_DIR

# ── 11 HIGH-QUALITY CLINICAL FEATURES (6 vitals removed) ──────────────────
# REMOVED: heart_rate, respiratory_rate, spo2, systolic_bp, diastolic_bp, temperature_f
# REASON : 78.3% missing in non-ICU scaleup patients (ICU-only chartevents table)
CLINICAL_FEATURES = [
    'age', 'gender_M', 'is_deceased',           # Demographics   (~95% coverage)
    'wbc', 'hemoglobin', 'hematocrit',           # Blood Panel    (~80% coverage)
    'creatinine',                                # Kidney Panel   (~80% coverage)
    'crp', 'alk_phos', 'albumin',               # Inflammatory   (~51-65% coverage)
    'has_vitals',                                # ICU Proxy Flag (100% coverage)
]
N_CLINICAL_FEATURES = len(CLINICAL_FEATURES)    # 11

print(f'Paired CSV    : {os.path.exists(PAIRED_CSV)} -> {PAIRED_CSV}')
print(f'BBox CSV      : {os.path.exists(BBOX_CSV)}')
print(f'P1 checkpoint : {os.path.exists(P1_CKPT)}')
print(f'P2v2 ckpt     : {os.path.exists(P2V2_CKPT)}')
print(f'Save dir      : {SAVE_DIR}')
print(f'Clinical feats: {N_CLINICAL_FEATURES} -> {CLINICAL_FEATURES}')


---
## Step 3: Data Loading
We load the single merged scaleup CSV which already contains all text, image paths, and clinical data in one file.


In [ ]:
# --- CELL 2: Data Loading (SCALEUP — single merged CSV) ---

def resolve_path(p):
    if pd.isna(p) or str(p).strip() == '': return ''
    p = str(p).replace('/', os.sep).replace('\\\\', os.sep)
    return p if os.path.isabs(p) else os.path.join(IMG_BASE, p)

# Load the single merged CSV (text + clinical already combined)
df_raw = pd.read_csv(PAIRED_CSV)
df_raw['image_path']  = df_raw['image_path'].apply(resolve_path)
df_raw['report_path'] = df_raw['report_path'].apply(resolve_path)

# Use raw_report column if available; otherwise read from disk
if 'raw_report' in df_raw.columns:
    df_raw['full_report'] = df_raw['raw_report'].fillna('').astype(str)
else:
    def read_report(path):
        try:
            with open(path, encoding='utf-8', errors='ignore') as f: return f.read().strip()
        except: return ''
    df_raw['full_report'] = df_raw['report_path'].apply(read_report)

print(f'Loaded {len(df_raw):,} rows | Labels: {df_raw["label"].value_counts().to_dict()}')
print(f'Clinical columns present: {[c for c in CLINICAL_FEATURES if c in df_raw.columns]}')
print(f'Missing features       : {[c for c in CLINICAL_FEATURES if c not in df_raw.columns]}')

# Check image availability
images_found = df_raw['image_path'].apply(os.path.exists).sum()
print(f'Images found on disk   : {images_found}/{len(df_raw)}')


---
## Step 4: Text Extraction (FINDINGS + HISTORY)
We extract only `FINDINGS` and `HISTORY` sections, suppressing `IMPRESSION` to prevent data leakage.


In [ ]:
# --- CELL 3: Text Extraction (FINDINGS + HISTORY) ---

# Alias for downstream cells
text_df = df_raw.copy()

LEAKAGE_RE = re.compile(
    r'\bpneumonia\b|\bpneumonic\b|\bno[ -]acute[ -]\w+|\bno finding\w*'
    r'|\bcompatible with\b|\bconsistent with\b|\bnormal study\b|\bno significant\b',
    re.IGNORECASE
)

def extract_rich_text(text):
    parts = []
    h = re.search(r'HISTORY[:\s]+(.*?)(?=FINDINGS|TECHNIQUE|COMPARISON|\n\n|\Z)',
                  text, re.DOTALL | re.IGNORECASE)
    if h: parts.append(h.group(1).strip())
    f = re.search(r'FINDINGS[:\s]+(.*?)(?=IMPRESSION|CONCLUSION|\n\n|\Z)',
                  text, re.DOTALL | re.IGNORECASE)
    if f: parts.append(f.group(1).strip())
    result = ' '.join(parts).strip()
    if not result: result = text[:512]
    result = LEAKAGE_RE.sub('[REDACTED]', result)
    return result

text_df['report_rich'] = text_df['full_report'].apply(extract_rich_text)

print(f'Reports processed : {len(text_df):,}')
print(f'Avg words (rich)  : {text_df["report_rich"].str.split().str.len().mean():.1f}')
empty = (text_df['report_rich'].str.strip() == '').sum()
print(f'Empty report_rich : {empty}')


---
## Step 5: Clinical Feature Validation & Missing Value Summary
We confirm the 11 selected features are present and display their missingness rates.


In [ ]:
# --- CELL 4: Clinical Feature Validation (SCALEUP 11-Feature) ---

df = text_df.copy()

print(f'Rows   : {len(df):,}')
print(f'\nLabel distribution:')
print(df['label'].value_counts().rename({0: 'Normal', 1: 'Pneumonia'}))

# Verify report_rich present
assert 'report_rich' in df.columns, "report_rich missing — re-run Cell 3"
print(f'\nreport_rich present : True')
print(f'Avg words (rich)    : {df["report_rich"].str.split().str.len().mean():.1f}')

# Verify 6 vitals are NOT in our feature list
vitals_excluded = ['heart_rate', 'respiratory_rate', 'spo2', 'systolic_bp', 'diastolic_bp', 'temperature_f']
print(f'\nVitals excluded from model: {vitals_excluded}')
print(f'has_vitals (ICU proxy) coverage: {df["has_vitals"].mean()*100:.1f}%')

# Check feature availability
missing_feats = [c for c in CLINICAL_FEATURES if c not in df.columns]
if missing_feats:
    print(f'WARNING - Missing features: {missing_feats}')
else:
    print(f'\nAll {N_CLINICAL_FEATURES} clinical features present.')

print(f'\nMissing values per retained feature (will be imputed in Cell 5):')
for col in CLINICAL_FEATURES:
    n   = df[col].isna().sum()
    bar = '#' * int((100 - 100*n/len(df)) / 5)
    print(f'  {col:25s}: {n:4d} ({100*n/len(df):.1f}%) {bar}')


---
## Step 6: Stratified Train / Val / Test Split & Leakage-Free Normalisation
Stratified 70 / 15 / 15 split. Missing values filled with **training set medians only**. `StandardScaler` fitted on training set only.


In [ ]:
# --- CELL 5: Train / Val / Test Split + NaN Imputation + Scaling ---

df = df[df['label'].notna()].reset_index(drop=True)
df['label'] = df['label'].astype(int)

CLINICAL_FEATURES_USED = [c for c in CLINICAL_FEATURES if c in df.columns]
N_CLINICAL_FEATURES    = len(CLINICAL_FEATURES_USED)
print(f'Clinical features: {N_CLINICAL_FEATURES} -> {CLINICAL_FEATURES_USED}')

# ── Stratified Split ───────────────────────────────────────────────────────
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED)
val_frac = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_frac, stratify=train_val_df['label'], random_state=SEED)

train_df = train_df.copy().reset_index(drop=True)
val_df   = val_df.copy().reset_index(drop=True)
test_df  = test_df.copy().reset_index(drop=True)

for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n0 = (split['label']==0).sum(); n1 = (split['label']==1).sum()
    print(f'{name:5s}: {len(split):5d}  (Normal={n0}, Pneumonia={n1})')

# ── Step 1: Median Imputation — TRAIN SET ONLY (No Leakage) ───────────────
print('\nImputing NaN with train-set median...')
train_medians = {}
for col in CLINICAL_FEATURES_USED:
    m = train_df[col].median()
    if pd.isna(m): m = 0.0
    train_medians[col] = m
    train_df[col] = train_df[col].fillna(m).astype(float)
    val_df[col]   = val_df[col].fillna(m).astype(float)
    test_df[col]  = test_df[col].fillna(m).astype(float)

total_nan = sum(
    train_df[c].isna().sum() + val_df[c].isna().sum() + test_df[c].isna().sum()
    for c in CLINICAL_FEATURES_USED)
print(f'NaN after imputation: {total_nan}  (must be 0)')
assert total_nan == 0

# ── Step 2: StandardScaler — Fit on Train Only ────────────────────────────
scaler = StandardScaler()
train_df[CLINICAL_FEATURES_USED] = scaler.fit_transform(train_df[CLINICAL_FEATURES_USED].astype(float))
val_df[CLINICAL_FEATURES_USED]   = scaler.transform(val_df[CLINICAL_FEATURES_USED].astype(float))
test_df[CLINICAL_FEATURES_USED]  = scaler.transform(test_df[CLINICAL_FEATURES_USED].astype(float))

total_nan_scaled = sum(train_df[c].isna().sum() for c in CLINICAL_FEATURES_USED)
print(f'NaN after scaling   : {total_nan_scaled}  (must be 0)')
assert total_nan_scaled == 0

# Update globals
CLINICAL_FEATURES   = CLINICAL_FEATURES_USED
N_CLINICAL_FEATURES = len(CLINICAL_FEATURES)
print(f'\nDone. N_CLINICAL_FEATURES = {N_CLINICAL_FEATURES}')
print('Clinical features standardised (fit on train only). No data leakage.')


---
## Step 7: Bounding Box Lookup & Multimodal Dataset Loader
We load lung bounding boxes and define `TripleModalCXRDataset` yielding `(image, text_ids, text_mask, clinical_vector, label)` per sample. CLAHE contrast enhancement is applied during image loading.


In [ ]:
# --- CELL 6: Bbox Lookup & Triple Dataset Class ---

bbox_df     = pd.read_csv(BBOX_CSV)
bbox_lookup = bbox_df.set_index('image_path').to_dict('index')
print(f'Bbox entries : {len(bbox_lookup)}')

tokenizer = AutoTokenizer.from_pretrained(CLINBERT_MODEL)

class TripleModalCXRDataset(Dataset):
    def __init__(self, df, bbox_lookup, tokenizer, img_transform, clinical_features, max_len=MAX_TEXT_LEN):
        self.df               = df.reset_index(drop=True)
        self.bbox_lookup      = bbox_lookup
        self.tokenizer        = tokenizer
        self.transform        = img_transform
        self.clinical_features = clinical_features
        self.max_len          = max_len
        self.clahe            = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # 1. Image Modality (with CLAHE + BBox crop)
        img = cv2.imread(row.image_path, cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((224, 224), dtype=np.uint8)
        img = self.clahe.apply(img)
        bb  = self.bbox_lookup.get(row.image_path, None)
        if bb and bb.get('x_max', 0) > 0:
            img = img[bb['y_min']:bb['y_max'], bb['x_min']:bb['x_max']]
        img = self.transform(Image.fromarray(img))

        # 2. Text Modality
        enc = self.tokenizer(
            row.report_rich, max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        # 3. Clinical Metadata (11 features)
        meta = torch.tensor(
            [float(row[c]) for c in self.clinical_features],
            dtype=torch.float32
        )

        return (
            img,
            enc['input_ids'].squeeze(0),
            enc['attention_mask'].squeeze(0),
            meta,
            int(row.label)
        )

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(8),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])
print('Dataset class ready.')


---
## Step 8: Network Architectures
- `DenseNet121CBAM`: Image Feature Extractor (1024-d).
- `TextEncoder`: Bio_ClinicalBERT (last 2 layers unfrozen, 768-d sequence output).
- `MetadataEncoder`: $11 \rightarrow 128 \rightarrow 128 \rightarrow 64$-d MLP.
- `TripleFusionNet`: CrossAttention($1024 + 512 + 64 = 1600$-d) + Classifier.


In [ ]:
# --- CELL 7: Model Architecture ---

class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1, bias=False), nn.ReLU(),
            nn.Conv2d(channels // reduction, channels, 1, bias=False), nn.Sigmoid()
        )
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid()
        )
    def forward(self, x):
        x = x * self.ca(x)
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sa(torch.cat([avg_out, max_out], dim=1))

class DenseNet121CBAM(nn.Module):
    def __init__(self):
        super().__init__()
        base = xrv.models.DenseNet(weights="densenet121-res224-all")
        self.features = base.model.features
        self.cbam     = CBAMBlock(1024)
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))
    def forward(self, x):
        feat = self.features(x)
        feat = self.cbam(feat)
        feat = self.pool(feat)
        return torch.flatten(feat, 1)  # (B, 1024)

class TextEncoder(nn.Module):
    def __init__(self, model_name=CLINBERT_MODEL):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in self.bert.parameters():
            param.requires_grad = False
        for param in self.bert.encoder.layer[-2:].parameters():
            param.requires_grad = True
    def forward(self, input_ids, attention_mask):
        return self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # (B, seq, 768)

class MetadataEncoder(nn.Module):
    def __init__(self, in_dim=N_CLINICAL_FEATURES, hidden=META_HIDDEN, out_dim=META_OUT_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, out_dim), nn.ReLU()
        )
    def forward(self, x):
        return self.net(x)  # (B, 64)

class TripleFusionNet(nn.Module):
    def __init__(self, img_dim=IMG_FEAT_DIM, txt_dim=TXT_FEAT_DIM,
                 attn_dim=ATTN_DIM, heads=ATTN_HEADS, meta_out=META_OUT_DIM):
        super().__init__()
        self.query_proj = nn.Linear(img_dim, attn_dim)
        self.key_proj   = nn.Linear(txt_dim, attn_dim)
        self.val_proj   = nn.Linear(txt_dim, attn_dim)
        self.cross_attn = nn.MultiheadAttention(attn_dim, heads, batch_first=True, dropout=0.1)
        self.norm1      = nn.LayerNorm(attn_dim)

        fused_dim = img_dim + attn_dim + meta_out  # 1024 + 512 + 64 = 1600
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim), nn.Dropout(0.4),
            nn.Linear(fused_dim, 512), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(512, 128),       nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, img_feat, txt_tokens, meta_feat):
        q = self.query_proj(img_feat).unsqueeze(1)
        k, v = self.key_proj(txt_tokens), self.val_proj(txt_tokens)
        attended, _ = self.cross_attn(q, k, v)
        attended = self.norm1(attended.squeeze(1))
        fused = torch.cat([img_feat, attended, meta_feat], dim=1)
        return self.classifier(fused)

print('All model architecture classes successfully defined.')


---
## Step 9: Focal Loss & Triple Mixup Utilities
**Focal Loss** ($\gamma = 2.0$) focuses on hard examples. **Triple Mixup** ($\alpha = 0.2$) augments across all three modalities simultaneously.


In [ ]:
# --- CELL 8: Focal Loss & Triple Mixup Utilities ---

class FocalLoss(nn.Module):
    def __init__(self, gamma=FOCAL_GAMMA, weight=None):
        super().__init__()
        self.gamma = gamma; self.weight = weight
    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt   = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()

def mixup_triple(img_f, txt_t, meta_f, labels, alpha=MIXUP_ALPHA):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx   = torch.randperm(img_f.size(0)).to(img_f.device)
    return (lam * img_f  + (1-lam) * img_f[idx],
            lam * txt_t  + (1-lam) * txt_t[idx],
            lam * meta_f + (1-lam) * meta_f[idx],
            labels, labels[idx], lam)

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


---
## Step 10: Model Instantiation & Pretrained Weight Loading
We load Phase 1.1v4 Scaleup (fold5) weights into the frozen image encoder, and Phase 2v2 Scaleup weights into the text encoder and fusion model.


In [ ]:
# --- CELL 9: Instantiate Models, Load Scaleup Weights, Build DataLoaders ---

train_ds = TripleModalCXRDataset(train_df, bbox_lookup, tokenizer, train_tfm, CLINICAL_FEATURES)
val_ds   = TripleModalCXRDataset(val_df,   bbox_lookup, tokenizer, val_tfm,   CLINICAL_FEATURES)
test_ds  = TripleModalCXRDataset(test_df,  bbox_lookup, tokenizer, val_tfm,   CLINICAL_FEATURES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

image_encoder = DenseNet121CBAM().to(DEVICE)
text_encoder  = TextEncoder().to(DEVICE)
meta_encoder  = MetadataEncoder(in_dim=N_CLINICAL_FEATURES).to(DEVICE)
fusion_model  = TripleFusionNet().to(DEVICE)

# 1. Load Phase 1.1v4 Scaleup Image Encoder (fold 5)
if os.path.exists(P1_CKPT):
    ckpt1 = torch.load(P1_CKPT, map_location=DEVICE)
    image_encoder.load_state_dict(ckpt1.get('model_state_dict', ckpt1), strict=False)
    print('Phase 1.1v4 Scaleup Image Encoder loaded (fold5).')
for p in image_encoder.parameters(): p.requires_grad = False

# 2. Load Phase 2v2 Scaleup Text & Fusion Weights
if os.path.exists(P2V2_CKPT):
    ckpt2 = torch.load(P2V2_CKPT, map_location=DEVICE)
    if 'text_encoder' in ckpt2: text_encoder.load_state_dict(ckpt2['text_encoder'], strict=False)
    if 'fusion_model' in ckpt2: fusion_model.load_state_dict(ckpt2['fusion_model'], strict=False)
    print('Phase 2v2 Scaleup Text & Fusion weights loaded.')

total_params = sum(p.numel() for p in meta_encoder.parameters())
print(f'MetadataEncoder parameters: {total_params:,}')
print(f'DataLoaders ready — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)} batches')


---
## Step 11: Training & Validation Loop
AdamW with 3 differential learning rates, Cosine Annealing LR, and Early Stopping (`patience=8`).


In [ ]:
# --- CELL 10: Training Loop ---

def eval_epoch(fusion, img_enc, txt_enc, meta_enc, loader, criterion, device):
    fusion.eval(); img_enc.eval(); txt_enc.eval(); meta_enc.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.no_grad():
        for imgs, ids, masks, meta, labels in loader:
            imgs, ids, masks, meta, labels = (
                imgs.to(device), ids.to(device), masks.to(device),
                meta.to(device), labels.to(device))
            img_f  = img_enc(imgs)
            txt_t  = txt_enc(ids, masks)
            meta_f = meta_enc(meta)
            logits = fusion(img_f, txt_t, meta_f)
            loss   = criterion(logits, labels)
            probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            total_loss += loss.item() * labels.size(0)
            all_probs.extend(probs); all_labels.extend(labels.cpu().tolist())
    acc = accuracy_score(all_labels, [1 if p >= 0.5 else 0 for p in all_probs])
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
    return total_loss / len(all_labels), acc, auc, all_probs, all_labels

def find_optimal_threshold(labels, probs):
    fpr, tpr, thresh = roc_curve(labels, probs)
    return float(thresh[np.argmax(tpr - fpr)])

def find_clinical_threshold(labels, probs, target_sens=0.90):
    fpr, tpr, thresh = roc_curve(labels, probs)
    for t, s in zip(thresh, tpr):
        if s >= target_sens: return float(t)
    return float(thresh[np.argmax(tpr - fpr)])

n0 = (train_df['label']==0).sum(); n1 = (train_df['label']==1).sum()
weight = torch.tensor([n1/(n0+n1), n0/(n0+n1)], dtype=torch.float).to(DEVICE)
criterion_focal = FocalLoss(gamma=FOCAL_GAMMA, weight=weight)

optimizer = torch.optim.AdamW([
    {'params': text_encoder.parameters(),  'lr': LR_BERT,   'weight_decay': 1e-4},
    {'params': fusion_model.parameters(),  'lr': LR_FUSION, 'weight_decay': 1e-4},
    {'params': meta_encoder.parameters(),  'lr': LR_META,   'weight_decay': 1e-3},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_auc, best_state, patience_cnt = 0.0, None, 0
history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': []}

for epoch in range(1, EPOCHS + 1):
    fusion_model.train(); image_encoder.eval(); text_encoder.train(); meta_encoder.train()
    ep_loss, ep_probs, ep_labels = 0.0, [], []

    for imgs, ids, masks, meta, labels in train_loader:
        imgs, ids, masks, meta, labels = (
            imgs.to(DEVICE), ids.to(DEVICE), masks.to(DEVICE),
            meta.to(DEVICE), labels.to(DEVICE))

        with torch.no_grad(): img_f = image_encoder(imgs)
        txt_t  = text_encoder(ids, masks)
        meta_f = meta_encoder(meta)

        img_m, txt_m, meta_m, la, lb, lam = mixup_triple(img_f, txt_t, meta_f, labels)

        optimizer.zero_grad()
        logits = fusion_model(img_m, txt_m, meta_m)
        loss   = mixup_criterion(criterion_focal, logits, la, lb, lam)
        loss.backward()

        all_params = (list(fusion_model.parameters()) +
                      list(text_encoder.parameters()) +
                      list(meta_encoder.parameters()))
        nn.utils.clip_grad_norm_(all_params, 1.0)
        optimizer.step()

        probs = F.softmax(logits.detach(), dim=1)[:, 1].cpu().numpy()
        ep_loss += loss.item() * labels.size(0)
        ep_probs.extend(probs); ep_labels.extend(labels.cpu().tolist())

    scheduler.step()
    tr_loss = ep_loss / len(ep_labels)
    tr_auc  = roc_auc_score(ep_labels, ep_probs) if len(set(ep_labels)) > 1 else 0.5

    val_loss, val_acc, val_auc, _, _ = eval_epoch(
        fusion_model, image_encoder, text_encoder, meta_encoder,
        val_loader, criterion_focal, DEVICE)

    history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss)
    history['train_auc'].append(tr_auc);   history['val_auc'].append(val_auc)

    marker = ''
    if val_auc > best_auc:
        best_auc   = val_auc
        best_state = copy.deepcopy({
            'fusion': fusion_model.state_dict(),
            'text':   text_encoder.state_dict(),
            'meta':   meta_encoder.state_dict()
        })
        torch.save(best_state, os.path.join(SAVE_DIR, 'best_p3_scaleup_11feat_model.pth'))
        patience_cnt = 0; marker = ' <-- best'
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or marker:
        print(f'Ep {epoch:03d} | tr_loss={tr_loss:.4f} tr_auc={tr_auc:.4f} | val_auc={val_auc:.4f} val_acc={val_acc:.3f}{marker}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

print(f'Best Val AUC: {best_auc:.4f}')


---
## Step 12: Test Set Evaluation & Decision Threshold Tuning
We evaluate across three operating thresholds: Default ($0.500$), Youden-J (optimal trade-off), and Clinical ($\ge 90\%$ Sensitivity).


In [ ]:
# --- CELL 11: Test Set Evaluation ---

if best_state is not None:
    fusion_model.load_state_dict(best_state['fusion'])
    text_encoder.load_state_dict(best_state['text'])
    meta_encoder.load_state_dict(best_state['meta'])
elif os.path.exists(os.path.join(SAVE_DIR, 'best_p3_scaleup_11feat_model.pth')):
    ckpt = torch.load(os.path.join(SAVE_DIR, 'best_p3_scaleup_11feat_model.pth'), map_location=DEVICE)
    fusion_model.load_state_dict(ckpt['fusion'])
    text_encoder.load_state_dict(ckpt['text'])
    meta_encoder.load_state_dict(ckpt['meta'])

test_loss, test_acc, test_auc, test_probs, test_labels = eval_epoch(
    fusion_model, image_encoder, text_encoder, meta_encoder,
    test_loader, criterion_focal, DEVICE)

thresh_youden   = find_optimal_threshold(test_labels, test_probs)
thresh_clinical = find_clinical_threshold(test_labels, test_probs, target_sens=0.90)

def calc_metrics(labels, probs, t):
    preds = [1 if p >= t else 0 for p in probs]
    cm_v  = confusion_matrix(labels, preds)
    tn, fp, fn, tp = cm_v.ravel()
    acc  = accuracy_score(labels, preds)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1   = f1_score(labels, preds)
    return acc, sens, spec, f1, preds

ac_d, s_d, sp_d, f1_d, p_d = calc_metrics(test_labels, test_probs, 0.500)
ac_y, s_y, sp_y, f1_y, p_y = calc_metrics(test_labels, test_probs, thresh_youden)
ac_c, s_c, sp_c, f1_c, p_c = calc_metrics(test_labels, test_probs, thresh_clinical)

print('='*65)
print(f'Phase 3 Scaleup (11 Features) — Test AUC: {test_auc:.4f}')
print('='*65)
print(f'Default   (0.500)          : Acc={ac_d:.3f} | Sens={s_d*100:.1f}% | Spec={sp_d*100:.1f}% | F1={f1_d:.3f}')
print(f'Youden-J  ({thresh_youden:.3f})        : Acc={ac_y:.3f} | Sens={s_y*100:.1f}% | Spec={sp_y*100:.1f}% | F1={f1_y:.3f}')
print(f'Clinical  ({thresh_clinical:.3f})        : Acc={ac_c:.3f} | Sens={s_c*100:.1f}% | Spec={sp_c*100:.1f}% | F1={f1_c:.3f}')
print('='*65)


---
## Step 13: Statistical Validation — Bootstrap 95% Confidence Intervals
1,000 bootstrap resamples on the test set to derive the $95\%$ CI for AUC.


In [ ]:
# --- CELL 12b: Bootstrap 95% Confidence Intervals ---

def bootstrap_auc_ci(labels, probs, n_bootstrap=1000, ci=0.95, seed=42):
    rng    = np.random.RandomState(seed)
    aucs   = []
    n      = len(labels)
    labels = np.array(labels)
    probs  = np.array(probs)
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        s_lb, s_pr = labels[idx], probs[idx]
        if len(np.unique(s_lb)) < 2: continue
        aucs.append(roc_auc_score(s_lb, s_pr))
    aucs  = np.array(aucs)
    alpha = (1 - ci) / 2
    return np.mean(aucs), np.percentile(aucs, 100*alpha), np.percentile(aucs, 100*(1-alpha))

mean_auc, ci_lower, ci_upper = bootstrap_auc_ci(test_labels, test_probs)

print('='*60)
print('  BOOTSTRAP 95% CONFIDENCE INTERVAL (AUC)')
print('='*60)
print(f'  Test AUC Point Estimate : {test_auc:.4f}')
print(f'  Bootstrap Mean AUC      : {mean_auc:.4f}')
print(f'  95% CI                  : [{ci_lower:.4f} – {ci_upper:.4f}]')
print(f'  CI Margin (±)           : ±{(ci_upper-ci_lower)/2:.4f}')
print('='*60)
print(f'\n  Paper Format: AUC = {test_auc:.3f} (95% CI: {ci_lower:.3f}–{ci_upper:.3f})')


---
## Step 14: Confusion Matrices
Across all three thresholds (Default, Youden-J, Clinical).


In [ ]:
# --- CELL 12: Confusion Matrices ---

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, preds, acc, sens, spec) in zip(axes, [
    ('Default (0.500)', p_d, ac_d, s_d, sp_d),
    (f'Youden-J ({thresh_youden:.3f})', p_y, ac_y, s_y, sp_y),
    (f'Clinical ({thresh_clinical:.3f})', p_c, ac_c, s_c, sp_c)
]):
    cm_m = confusion_matrix(test_labels, preds)
    sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, annot_kws={'size': 14})
    ax.set_title(f'{title}\nAcc: {acc*100:.1f}% | Sens: {sens*100:.1f}% | Spec: {spec*100:.1f}%', fontsize=12)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Phase 3 Scaleup (11 Features) — Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3_scaleup_11feat_confusion_matrices.png'), dpi=150)
plt.show()


---
## Step 15: ROC Curve


In [ ]:
# --- CELL 13: ROC Curve ---

fpr, tpr, _ = roc_curve(test_labels, test_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkred', lw=2.5,
         label=f'Phase 3 Scaleup 11-Feat (AUC={test_auc:.4f} [{ci_lower:.3f}-{ci_upper:.3f}])')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.scatter([1-sp_y], [s_y], color='blue',  s=80, zorder=5, label=f'Youden-J ({thresh_youden:.3f})')
plt.scatter([1-sp_c], [s_c], color='green', s=80, zorder=5, label=f'Clinical 90% Sens ({thresh_clinical:.3f})')

plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Phase 3 Scaleup (11 Features) — Test Set ROC Curve')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3_scaleup_11feat_roc_curve.png'), dpi=150)
plt.show()


---
## Step 16: Learning & Loss Convergence Curves


In [ ]:
# --- CELL 14: Training Curves ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_auc'], label='Train AUC', color='royalblue')
axes[0].plot(history['val_auc'],   label='Val AUC',   color='darkorange')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].set_title('AUC Progression'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_loss'], label='Train Loss', color='royalblue')
axes[1].plot(history['val_loss'],   label='Val Loss',   color='darkorange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Focal Loss')
axes[1].set_title('Loss Progression'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Phase 3 Scaleup (11 Features) — Training History', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3_scaleup_11feat_training_curves.png'), dpi=150)
plt.show()


---
## Step 17: Multi-Phase Performance Progression Comparison
Comparing Phase 1, Phase 2, Phase 3 (17 features), and Phase 3 (11 clean features) on the Scaleup Dataset.


In [ ]:
# --- CELL 15: All-Phases Comparison Chart (Scaleup) ---

phases = [
    {'name': 'Phase 1\nImage only\n(Scaleup fold5)',    'auc': 0.850,  'sens': None,  'spec': None},
    {'name': 'Phase 2\nImg+Text\n(Scaleup CrossAttn)',  'auc': 0.9460, 'sens': 86.9,  'spec': 89.7},
    {'name': 'Phase 3\nImg+Text+Meta\n(17 feats)',      'auc': 0.9690, 'sens': 89.8,  'spec': 93.8},
    {'name': 'Phase 3\nImg+Text+Meta\n(11 feats ✓)',    'auc': round(test_auc, 4),
                                                          'sens': round(s_y*100, 1),
                                                          'spec': round(sp_y*100, 1)},
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
names  = [p['name'] for p in phases]
colors = ['#4477AA', '#66CCEE', '#228833', '#EE6677']

# AUC
axes[0].bar(names, [p['auc'] for p in phases], color=colors, edgecolor='black', alpha=0.85)
axes[0].set_ylim(0.80, 1.0)
axes[0].set_ylabel('AUC'); axes[0].set_title('Test AUC')
axes[0].grid(axis='y', alpha=0.3)
for i, p in enumerate(phases):
    axes[0].text(i, p['auc']+0.001, f"{p['auc']:.4f}", ha='center', fontsize=10, fontweight='bold')

# Sensitivity
sens_vals = [p['sens'] if p['sens'] is not None else 0 for p in phases]
axes[1].bar(names[1:], [v for v in sens_vals[1:]], color=colors[1:], edgecolor='black', alpha=0.85)
axes[1].set_ylim(75, 100)
axes[1].axhline(90, ls='--', color='red', alpha=0.6, label='90% target')
axes[1].set_ylabel('Sensitivity (%)'); axes[1].set_title('Sensitivity (Youden-J threshold)')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(sens_vals[1:]):
    axes[1].text(i, v+0.4, f"{v:.1f}%", ha='center', fontsize=10, fontweight='bold')

# Specificity
spec_vals = [p['spec'] if p['spec'] is not None else 0 for p in phases]
axes[2].bar(names[1:], [v for v in spec_vals[1:]], color=colors[1:], edgecolor='black', alpha=0.85)
axes[2].set_ylim(75, 100)
axes[2].set_ylabel('Specificity (%)'); axes[2].set_title('Specificity (Youden-J threshold)')
axes[2].grid(axis='y', alpha=0.3)
for i, v in enumerate(spec_vals[1:]):
    axes[2].text(i, v+0.4, f"{v:.1f}%", ha='center', fontsize=10, fontweight='bold')

plt.suptitle('PneumoFusionNet — Scaleup Dataset All-Phases Comparison', fontsize=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3_scaleup_11feat_all_phases_comparison.png'), bbox_inches='tight', dpi=150)
plt.show()


---
## Step 18: Clinical Feature Importance Analysis (Permutation Test)
AUC drop when each feature is shuffled — larger drop = more important feature.


In [ ]:
# --- CELL 16: Clinical Feature Importance (Permutation-based) ---

print('Computing clinical feature importances via permutation...')
fusion_model.eval(); image_encoder.eval(); text_encoder.eval(); meta_encoder.eval()

_, _, baseline_auc, _, _ = eval_epoch(
    fusion_model, image_encoder, text_encoder, meta_encoder,
    test_loader, criterion_focal, DEVICE)
print(f'Baseline AUC: {baseline_auc:.4f}')

feature_importances = {}
for feat_idx, feat_name in enumerate(CLINICAL_FEATURES):
    all_probs_perm, all_labels_perm = [], []
    with torch.no_grad():
        for imgs, ids, masks, meta, labels in test_loader:
            imgs, ids, masks, meta, labels = (
                imgs.to(DEVICE), ids.to(DEVICE), masks.to(DEVICE),
                meta.clone().to(DEVICE), labels.to(DEVICE))
            perm_idx = torch.randperm(meta.size(0))
            meta[:, feat_idx] = meta[perm_idx, feat_idx]
            img_f  = image_encoder(imgs)
            txt_t  = text_encoder(ids, masks)
            meta_f = meta_encoder(meta)
            logits = fusion_model(img_f, txt_t, meta_f)
            probs  = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs_perm.extend(probs); all_labels_perm.extend(labels.cpu().tolist())
    perm_auc = roc_auc_score(all_labels_perm, all_probs_perm)
    feature_importances[feat_name] = baseline_auc - perm_auc

sorted_feats = sorted(feature_importances.items(), key=lambda x: x[1], reverse=True)
feat_names   = [f[0] for f in sorted_feats]
feat_imps    = [f[1] for f in sorted_feats]

plt.figure(figsize=(10, 6))
colors_bar = ['#EE6677' if v > 0 else '#4477AA' for v in feat_imps]
plt.barh(feat_names, feat_imps, color=colors_bar, edgecolor='black', alpha=0.85)
plt.axvline(0, color='black', lw=1)
plt.xlabel('AUC Drop when Feature Permuted (higher = more important)')
plt.title('Clinical Feature Importance (Permutation-based)\nPhase 3 Scaleup — 11 Features')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'p3_scaleup_11feat_feature_importance.png'), dpi=150)
plt.show()

print('\nFeature Importances:')
for name, imp in sorted_feats:
    print(f'  {name:20s}: {imp:+.4f}')


---
## Step 19: Save Final Results JSON & Experiment Metadata


In [ ]:
# --- CELL 17: Save Results JSON & Config ---

results = {
    'phase': 3,
    'version': 'scaleup_11features',
    'description': 'Triple Fusion: Image + Text + 11 Clinical Features (Scaleup Dataset)',
    'architecture': {
        'image_encoder':  'DenseNet121+CBAM (frozen from Phase 1.1v4 Scaleup Fold 5)',
        'text_encoder':   'Bio_ClinicalBERT (last 2 layers unfrozen)',
        'meta_encoder':   f'MLP({N_CLINICAL_FEATURES}->128->128->{META_OUT_DIM})',
        'fusion':         'CrossAttention(img queries text) + concat meta + MLP classifier',
        'fused_dim':      IMG_FEAT_DIM + ATTN_DIM + META_OUT_DIM,
        'clinical_feats': CLINICAL_FEATURES,
        'vitals_removed': ['heart_rate', 'respiratory_rate', 'spo2',
                           'systolic_bp', 'diastolic_bp', 'temperature_f'],
        'vitals_removal_reason': '78.3% missing in non-ICU scaleup patients (ICU-only chartevents)'
    },
    'training': {
        'n_train': len(train_df), 'n_val': len(val_df), 'n_test': len(test_df),
        'batch_size': BATCH_SIZE, 'epochs_run': len(history['val_auc']),
        'lr_fusion': LR_FUSION, 'lr_bert': LR_BERT, 'lr_meta': LR_META,
        'focal_gamma': FOCAL_GAMMA, 'mixup_alpha': MIXUP_ALPHA
    },
    'results': {
        'test_auc':             round(test_auc, 4),
        'auc_ci_lower':         round(ci_lower, 4),
        'auc_ci_upper':         round(ci_upper, 4),
        'best_val_auc':         round(best_auc, 4),
        'youden_threshold':     round(thresh_youden, 4),
        'youden_acc':           round(ac_y, 4),
        'youden_sensitivity':   round(s_y, 4),
        'youden_specificity':   round(sp_y, 4),
        'clinical_threshold':   round(thresh_clinical, 4),
        'clinical_acc':         round(ac_c, 4),
        'clinical_sensitivity': round(s_c, 4),
        'clinical_specificity': round(sp_c, 4),
        'n_test': len(test_labels)
    },
    'comparison_vs_17feat_scaleup': {
        'p3_17feat_auc': 0.9690,
        'p3_11feat_auc': round(test_auc, 4),
        'auc_delta':     round(test_auc - 0.9690, 4)
    },
    'feature_importances': {k: round(v, 4) for k, v in feature_importances.items()}
}

out_path = os.path.join(SAVE_DIR, 'phase3_scaleup_11feat_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

print('='*65)
print('PHASE 3 SCALEUP (11 FEATURES) — FINAL RESULTS')
print('='*65)
print(f'Test AUC            : {test_auc:.4f} (95% CI: {ci_lower:.4f}–{ci_upper:.4f})')
print(f'Sensitivity (Y-J)   : {s_y*100:.1f}%')
print(f'Specificity (Y-J)   : {sp_y*100:.1f}%')
print(f'Accuracy    (Y-J)   : {ac_y*100:.1f}%')
print(f'vs 17-feature model : {test_auc - 0.9690:+.4f} AUC delta')
print(f'Results saved to    : {out_path}')
